# Setup Notebook Environment

This notebook is Colab-ready. It mounts Google Drive, installs dependencies, prepares the dataset, fine-tunes YOLOv8, saves weights back to Drive, and exports detection metadata for an XGBoost risk model.

In [ ]:
```python
# Install dependencies (run once in Colab)
!pip install -q ultralytics gdown xgboost scikit-learn opencv-python-headless pyyaml joblib tqdm

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set working directories
import os
WORK_DIR = '/content/drive/MyDrive/sentinel_model'  # change if you prefer a different path in Drive
os.makedirs(WORK_DIR, exist_ok=True)
%cd /content
```


In [ ]:
# Define the 7 To-Do Items
todos = {
    1: 'Create Jupyter/Colab training notebook',
    2: 'Mount Google Drive / fetch dataset',
    3: 'Prepare YOLO data.yaml and dataset checks',
    4: 'Install Python deps and run YOLO fine-tune',
    5: 'Save/export trained weights to Drive',
    6: 'Prototype XGBoost risk model from detection metadata',
    7: 'Package artifacts and plan backend integration',
}

for k, v in todos.items():
    print(k, v)


In [ ]:
# Implement To-Do Item 1: Verify environment and helper scripts
import os
from pathlib import Path
print('Working directory:', os.getcwd())
print('Drive path for artifacts:', WORK_DIR)

# Optionally, copy helper scripts from the repository (if you uploaded them to Drive)
# If you cloned the repo into Drive, adjust paths accordingly.


In [ ]:
# Implement To-Do Item 2: Mount Drive & optionally download dataset

# If your dataset is in Drive, define the Drive path to it (change this if needed)
DRIVE_DATA_PATH = '/content/drive/MyDrive/sentinel_dataset'  # user: adjust

if Path(DRIVE_DATA_PATH).exists():
    print('Dataset folder found in Drive:', DRIVE_DATA_PATH)
else:
    print('Drive dataset folder not found. You can either upload it to Drive or use gdown (see next cell).')

# Optional: Use gdown to download if a public direct link is available (may fail for restricted Drive folders)
# from google.colab import files
# !gdown --folder 'https://drive.google.com/drive/folders/14Z6DaVJEDst1coa-kA6V1tM0qEfTqoH8' -O ./data


In [ ]:
# Implement To-Do Item 3: Prepare YOLO data.yaml and dataset checks
import yaml

def generate_data_yaml(data_root, output_path='/content/data/data.yaml'):
    from pathlib import Path
    data_root = Path(data_root)
    # common candidate paths
    candidates = [
        (data_root / 'train' / 'images', data_root / 'val' / 'images'),
        (data_root / 'images' / 'train', data_root / 'images' / 'val'),
    ]
    train, val = None, None
    for t, v in candidates:
        if t.exists() and v.exists():
            train, val = t, v
            break
    if not train or not val:
        raise FileNotFoundError('Could not locate train/val image folders under ' + str(data_root))

    # attempt to find names file
    names = []
    for name_file in ['classes.txt','names.txt','data.names']:
        p = data_root / name_file
        if p.exists():
            names = [l.strip() for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]
            break

    data = {'train': str(train.resolve()), 'val': str(val.resolve()), 'nc': len(names), 'names': names}
    yaml_path = Path(output_path)
    yaml_path.parent.mkdir(parents=True, exist_ok=True)
    yaml.safe_dump(data, yaml_path, sort_keys=False)
    print('Wrote data.yaml to', yaml_path)
    return data

# Example usage (adjust DRIVE_DATA_PATH if needed)
# data_cfg = generate_data_yaml('/content/data')


In [ ]:
# Implement To-Do Item 4: Install Python deps (already installed above) and run a short YOLOv8 fine-tune
from ultralytics import YOLO

# small demo training run - set epochs=1 or 2 for quick test
# Ensure data.yaml path points to your prepared config
DATA_YAML = '/content/data/data.yaml'  # update if you generated it in a different path

def quick_train(pretrained='yolo8n.pt', epochs=1, batch=8, imgsz=640):
    print('Starting quick training...')
    model = YOLO(pretrained)
    model.train(data=DATA_YAML, epochs=epochs, batch=batch, imgsz=imgsz, project=WORK_DIR, name='sentinel_quick', device='0')
    print('Training finished')

# To run (uncomment):
# quick_train(epochs=1, batch=8)


In [ ]:
# Implement To-Do Item 5: Save/export trained weights to Drive

# The ultralytics training run saves checkpoints in the project folder (WORK_DIR). After training, copy best.pt
import shutil

def export_best_weights(run_name='sentinel_quick'):
    run_dir = Path(WORK_DIR) / 'runs' / 'train' / run_name
    candidates = list(run_dir.glob('**/best*.pt')) + list(run_dir.glob('**/weights/*.pt'))
    if not candidates:
        print('No weights found in', run_dir)
        return None
    best = candidates[0]
    dest = Path(WORK_DIR) / best.name
    shutil.copy(best, dest)
    print('Copied best weights to', dest)
    return str(dest)

# Implement To-Do 6: Run quick detection on a sample image and save JSON metadata for XGBoost
import json

def run_detection_and_save(model_path, sample_image='/content/sample.jpg', out_json='/content/detections.json'):
    model = YOLO(model_path)
    results = model.predict(source=sample_image, imgsz=640, conf=0.25)
    detections = []
    for r in results:
        boxes = r.boxes
        for b in boxes:
            det = {
                'xyxy': b.xyxy.tolist(),
                'xywh': b.xywh.tolist(),
                'conf': float(b.conf.tolist()[0]),
                'cls': int(b.cls.tolist()[0])
            }
            detections.append(det)
    with open(out_json, 'w') as f:
        json.dump(detections, f)
    print('Saved detection metadata to', out_json)
    return out_json

# Note: set sample_image to a real image path accessible in Colab or Drive


In [ ]:
# Implement To-Do Item 6: Prototype XGBoost training outline (from detection metadata)
# This is a small example that would load detections.json and train a toy XGBoost model on engineered features
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb

def train_risk_model_from_detections(detections_json='/content/detections.json'):
    import json
    with open(detections_json) as f:
        dets = json.load(f)
    # Example feature engineering: count defects, mean confidence, mean area
    if not dets:
        print('No detections found.')
        return None
    df = pd.DataFrame(dets)
    # create toy features assuming xywh present
    df['area'] = df['xywh'].apply(lambda x: x[2]*x[3])
    features = df.groupby(df.index).agg({'conf':'mean','area':'mean'}).reset_index(drop=True)
    # create toy target (synthetic) for demo: higher conf+area -> higher risk
    features['risk'] = (features['conf']*0.6 + (features['area']/features['area'].max())*0.4) * 100
    X = features[['conf','area']]
    y = features['risk']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)
    params = {'objective':'reg:squarederror', 'eval_metric':'rmse'}
    bst = xgb.train(params, dtrain, num_boost_round=50)
    preds = bst.predict(dtest)
    print('Example RMSE:', ((preds - y_test)**2).mean()**0.5)
    # save model
    bst.save_model('/content/risk_model.json')
    print('Saved XGBoost model to /content/risk_model.json')
    return '/content/risk_model.json'

# Implement To-Do 7: Package artifacts and plan backend integration (explain next steps)


In [ ]:
# Section: Validate and Persist Results
from pathlib import Path

def validate_artifacts():
    artifacts = {
        'data_yaml': Path('/content/data/data.yaml'),
        'weights': Path(WORK_DIR) / 'best.pt',
        'detections': Path('/content/detections.json'),
        'risk_model': Path('/content/risk_model.json')
    }
    for k, p in artifacts.items():
        print(k, '->', p.exists())

validate_artifacts()

print('Notebook cells inserted. Run cells interactively in Colab or local Jupyter.')


In [ ]:
# Auto-detect dataset folder in your mounted Google Drive
import os
from pathlib import Path

MYDRIVE = Path('/content/drive/MyDrive')
DRIVE_DATA_PATH = None
if MYDRIVE.exists():
    # look for folders containing 'sentinel' or 'merged' (case-insensitive)
    for root, dirs, files in os.walk(MYDRIVE):
        for d in dirs:
            dn = d.lower()
            if 'sentinel' in dn or 'merged' in dn:
                DRIVE_DATA_PATH = os.path.join(root, d)
                break
        if DRIVE_DATA_PATH:
            break

if DRIVE_DATA_PATH:
    print('Auto-detected dataset folder at:', DRIVE_DATA_PATH)
else:
    print('No sentinel/merged dataset folder found under /content/drive/MyDrive.')
    print('If the dataset is in "Shared with me", add a shortcut to My Drive then re-run the mount cell.')

# Use DRIVE_DATA_PATH in subsequent cells if not None
